# Week 10: Hyper-Contextual Dataset Generation
## ECE Model Inference + Full ESConv Processing (Days 9-10)

**Objective:** Generate a comprehensive instruction-tuning dataset for LLM fine-tuning by:
1. Running our trained ECE model over the entire ESConv dataset
2. Enriching each conversation turn with context, emotions, causes, and entities
3. Formatting everything for instruction-based fine-tuning

**Pipeline:**
- **Day 9:** ECE Model Inference Testing
- **Day 10:** Full Dataset Processing & Hyper-Contextual Enrichment

**Author:** Aura ML Project  
**Date:** November 16, 2025

---

## 📦 Install Required Libraries

Install all necessary libraries for ECE inference, NER, and dataset processing.

In [ ]:
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
%pip install transformers
%pip install spacy
%pip install tqdm

# Download spaCy model for NER
!python -m spacy download en_core_web_sm

## 🔧 Import Libraries and Setup

In [ ]:
import json
import torch
import torch.nn as nn
from pathlib import Path
from typing import List, Dict, Tuple, Optional
from tqdm.auto import tqdm
import random
import spacy

# Transformers
from transformers import (
    RobertaTokenizer,
    RobertaModel,
    RobertaPreTrainedModel,
    RobertaConfig
)

# Set random seeds
torch.manual_seed(42)
random.seed(42)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Using device: {device}")
print(f"✅ PyTorch version: {torch.__version__}")

# Load spaCy for NER
print("\nLoading spaCy model for NER...")
nlp = spacy.load("en_core_web_sm")
print("✅ spaCy model loaded successfully!")

---
## 🤖 Part 1: ECE Model Inference Testing (Day 9)

### 1.1 Define ECE Model Architecture

Re-define the ECEModel class to load our trained model.

In [ ]:
class ECEModel(RobertaPreTrainedModel):
    """Two-stage ECE model with clause classifier and span extractor."""
    
    def __init__(self, config):
        super().__init__(config)
        self.roberta = RobertaModel(config)
        
        # Clause classification head (binary: has cause or not)
        self.clause_classifier = nn.Sequential(
            nn.Dropout(0.1),
            nn.Linear(config.hidden_size, 1)
        )
        
        # Span extraction head (BIO tagging: O, B-CAUSE, I-CAUSE)
        self.span_classifier = nn.Sequential(
            nn.Dropout(0.1),
            nn.Linear(config.hidden_size, 3)  # 3 labels: O, B-CAUSE, I-CAUSE
        )
        
        self.init_weights()
    
    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        bio_labels=None,
        clause_label=None
    ):
        # Get RoBERTa outputs
        outputs = self.roberta(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        
        sequence_output = outputs.last_hidden_state  # [batch, seq_len, hidden_size]
        pooled_output = outputs.pooler_output  # [batch, hidden_size]
        
        # Clause classification (using [CLS] token)
        clause_logits = self.clause_classifier(pooled_output)  # [batch, 1]
        
        # Span extraction (token-level classification)
        span_logits = self.span_classifier(sequence_output)  # [batch, seq_len, 3]
        
        outputs_dict = {
            'clause_logits': clause_logits,
            'span_logits': span_logits
        }
        
        # Calculate losses if labels provided (training mode)
        if bio_labels is not None and clause_label is not None:
            # Clause loss (binary cross-entropy)
            clause_loss_fct = nn.BCEWithLogitsLoss()
            clause_loss = clause_loss_fct(
                clause_logits.squeeze(-1),
                clause_label.float()
            )
            
            # Span loss (cross-entropy for token classification)
            span_loss_fct = nn.CrossEntropyLoss(ignore_index=-100)
            
            # Reshape for loss calculation
            active_loss = attention_mask.view(-1) == 1
            active_logits = span_logits.view(-1, 3)[active_loss]
            active_labels = bio_labels.view(-1)[active_loss]
            
            span_loss = span_loss_fct(active_logits, active_labels)
            
            # Combined loss (weighted)
            total_loss = 0.3 * clause_loss + 0.7 * span_loss
            
            outputs_dict.update({
                'loss': total_loss,
                'clause_loss': clause_loss,
                'span_loss': span_loss
            })
        
        return outputs_dict

print("✅ ECEModel class defined successfully!")

### 1.2 Load Trained ECE Model

Load the best checkpoint from Week 9 training.

In [ ]:
# Paths
MODEL_PATH = Path("models/ece_roberta_best.pt")

# Load tokenizer
print("Loading RoBERTa tokenizer...")
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
print("✅ Tokenizer loaded!")

# Load model
print("\nLoading trained ECE model...")
config = RobertaConfig.from_pretrained('roberta-base')
model = ECEModel(config)

# Load checkpoint
if MODEL_PATH.exists():
    checkpoint = torch.load(MODEL_PATH, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
    model.eval()
    
    print(f"✅ Model loaded from: {MODEL_PATH}")
    print(f"   Best F1 Score: {checkpoint.get('best_f1', 'N/A')}")
    print(f"   Epoch: {checkpoint.get('epoch', 'N/A')}")
else:
    print(f"⚠️  Model file not found: {MODEL_PATH}")
    print("   Please ensure you've trained the model in Week 9 first.")
    raise FileNotFoundError(f"Model checkpoint not found at {MODEL_PATH}")

### 1.3 Create Inference Function

Implement the `find_cause()` function that extracts causes from text using the trained model.

In [ ]:
def find_cause(text: str, max_length: int = 128) -> str:
    """
    Extract emotion cause from text using trained ECE model.
    
    Args:
        text: Input text to analyze
        max_length: Maximum sequence length for tokenization
        
    Returns:
        Extracted cause string (or full text if no cause detected)
    """
    # Tokenize input
    inputs = tokenizer(
        text,
        max_length=max_length,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    
    input_ids = inputs['input_ids'].to(device)
    attention_mask = inputs['attention_mask'].to(device)
    
    # Run inference
    with torch.no_grad():
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
    
    # Get predicted BIO tags
    span_logits = outputs['span_logits']  # [1, seq_len, 3]
    predicted_tags = torch.argmax(span_logits, dim=-1)[0]  # [seq_len]
    
    # BIO label mapping
    id2label = {0: 'O', 1: 'B-CAUSE', 2: 'I-CAUSE'}
    
    # Convert to tokens
    tokens = tokenizer.convert_ids_to_tokens(input_ids[0].cpu().numpy())
    
    # Extract cause tokens (B-CAUSE and I-CAUSE)
    cause_tokens = []
    for token, tag_id in zip(tokens, predicted_tags.cpu().numpy()):
        tag = id2label[tag_id]
        if tag in ['B-CAUSE', 'I-CAUSE']:
            cause_tokens.append(token)
    
    # Decode cause tokens to string
    if cause_tokens:
        # Remove special tokens and clean up
        cause_tokens = [t for t in cause_tokens if t not in ['<s>', '</s>', '<pad>']]
        
        # Join and clean up RoBERTa's Ġ prefix (space indicator)
        cause_text = tokenizer.convert_tokens_to_string(cause_tokens)
        cause_text = cause_text.strip()
        
        return cause_text if cause_text else text
    else:
        # Fallback: No cause detected, return full text
        return text

print("✅ find_cause() function created successfully!")

### 1.4 Test Cases - Standalone Inference

Test the ECE model on various example sentences.

In [ ]:
# Test cases
test_sentences = [
    "I feel sad because I lost my job yesterday.",
    "I'm anxious about my upcoming exam next week.",
    "I'm angry because my friend betrayed my trust.",
    "I feel happy since I got accepted into my dream university.",
    "I'm scared because I have to give a presentation in front of 100 people.",
    "I feel lonely after moving to a new city where I don't know anyone.",
    "I'm frustrated because my computer keeps crashing.",
    "I feel guilty about forgetting my mom's birthday.",
    "I'm disappointed that my team lost the championship game.",
    "I feel overwhelmed with all the work I have to do this week."
]

print(f"{'='*80}")
print(f"ECE MODEL INFERENCE TEST CASES")
print(f"{'='*80}\n")

for idx, sentence in enumerate(test_sentences, 1):
    cause = find_cause(sentence)
    
    print(f"Test {idx}:")
    print(f"  Input:  {sentence}")
    print(f"  Cause:  {cause}")
    print()

print(f"{'='*80}")
print("✅ All test cases completed!")
print(f"{'='*80}")

---
## 🚀 Part 2: Hyper-Contextual Dataset Generation (Day 10)

### 2.1 Load ESConv Dataset

Load the full ESConv dataset for processing.

In [ ]:
# Define paths to ESConv dataset
DATASET_DIR = Path("esconv_dataset")
TRAIN_FILE = DATASET_DIR / "train.jsonl"
VAL_FILE = DATASET_DIR / "validation.jsonl"
TEST_FILE = DATASET_DIR / "test.jsonl"

def load_esconv_dataset(file_path: Path) -> List[Dict]:
    """Load ESConv dataset from JSONL file."""
    conversations = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                conversations.append(json.loads(line))
            except json.JSONDecodeError as e:
                print(f"Error decoding JSON: {e}")
    return conversations

# Load all splits
print("Loading ESConv dataset...")
train_data = load_esconv_dataset(TRAIN_FILE)
val_data = load_esconv_dataset(VAL_FILE)
test_data = load_esconv_dataset(TEST_FILE)

# Combine all conversations
all_conversations = train_data + val_data + test_data

print(f"\n{'='*60}")
print(f"📊 DATASET STATISTICS")
print(f"{'='*60}")
print(f"Training conversations:   {len(train_data):,}")
print(f"Validation conversations: {len(val_data):,}")
print(f"Test conversations:       {len(test_data):,}")
print(f"{'='*60}")
print(f"Total conversations:      {len(all_conversations):,}")
print(f"{'='*60}\n")

print(f"✅ Loaded {len(all_conversations):,} conversations")

### 2.2 Define Support Strategies

List all available support strategies for the instruction template.

In [ ]:
# ESConv support strategies
SUPPORT_STRATEGIES = [
    "Question",
    "Restatement or Paraphrasing",
    "Reflection of feelings",
    "Self-disclosure",
    "Affirmation and Reassurance",
    "Providing Suggestions",
    "Information",
    "Others"
]

strategies_str = ", ".join(SUPPORT_STRATEGIES)
print(f"✅ Support strategies defined: {len(SUPPORT_STRATEGIES)} strategies")
print(f"   {strategies_str}")

### 2.3 Helper Functions

Create helper functions for NER and context management.

In [ ]:
def extract_entities(text: str) -> List[str]:
    """
    Extract named entities from text using spaCy.
    
    Args:
        text: Input text
        
    Returns:
        List of entity strings with their labels
    """
    doc = nlp(text)
    entities = [f"{ent.text} ({ent.label_})" for ent in doc.ents]
    return entities


def format_history(history: List[Dict]) -> str:
    """
    Format conversation history into a readable string.
    
    Args:
        history: List of conversation turns
        
    Returns:
        Formatted history string
    """
    if not history:
        return "[No previous context]"
    
    formatted = []
    for turn in history:
        speaker = "User" if turn['speaker'] == 'seeker' else "Aura"
        formatted.append(f"{speaker}: {turn['text']}")
    
    return " | ".join(formatted)


def infer_problem_type(text: str, emotion: str) -> str:
    """
    Infer problem type based on text content and emotion.
    
    Args:
        text: User's message
        emotion: Detected emotion
        
    Returns:
        Problem type category
    """
    text_lower = text.lower()
    
    # Relationship keywords
    if any(word in text_lower for word in ['friend', 'relationship', 'partner', 'family', 'boyfriend', 'girlfriend', 'husband', 'wife']):
        return "relationship"
    
    # Work/career keywords
    elif any(word in text_lower for word in ['job', 'work', 'career', 'boss', 'colleague', 'interview', 'promotion']):
        return "work"
    
    # Health keywords
    elif any(word in text_lower for word in ['health', 'sick', 'illness', 'hospital', 'doctor', 'pain']):
        return "health"
    
    # Academic keywords
    elif any(word in text_lower for word in ['school', 'exam', 'test', 'study', 'university', 'college', 'grade']):
        return "academic"
    
    # Financial keywords
    elif any(word in text_lower for word in ['money', 'debt', 'financial', 'rent', 'bills', 'poor']):
        return "financial"
    
    # Default based on emotion
    elif emotion in ['sad', 'fear', 'angry']:
        return "emotional_distress"
    
    else:
        return "general"

print("✅ Helper functions defined successfully!")

### 2.4 Core Pipeline Function

Implement the main processing function that enriches conversations with ECE, NER, and context.

In [ ]:
def process_conversation(conversation: Dict) -> List[Dict]:
    """
    Process a single conversation and generate instruction-tuning samples.
    
    For each seeker turn:
    1. Maintain sliding window history (last 3 turns)
    2. Extract emotion from ESConv labels
    3. Run ECE model to find cause
    4. Run spaCy NER to extract entities
    5. Get supporter's response as target
    6. Format as instruction-tuning sample
    
    Args:
        conversation: ESConv conversation dictionary
        
    Returns:
        List of instruction-tuning samples
    """
    samples = []
    dialog = conversation.get('dialog', [])
    
    # Sliding window history (last 3 turns)
    history = []
    HISTORY_WINDOW = 3
    
    for idx, utterance in enumerate(dialog):
        # Only process seeker utterances
        if utterance.get('speaker') != 'seeker':
            # Add to history and continue
            history.append({
                'speaker': 'supporter',
                'text': utterance.get('text', '')
            })
            # Keep only last N turns
            history = history[-HISTORY_WINDOW:]
            continue
        
        # Get seeker's message
        user_message = utterance.get('text', '').strip()
        if not user_message:
            continue
        
        # Get emotion (with fallback)
        emotion = utterance.get('emotion', 'neutral')
        if not emotion:
            emotion = 'neutral'
        
        # Run ECE model to extract cause
        extracted_cause = find_cause(user_message)
        
        # Run NER to extract entities
        entities = extract_entities(user_message)
        entities_str = ", ".join(entities) if entities else "None"
        
        # Infer problem type
        problem_type = infer_problem_type(user_message, emotion)
        
        # Format conversation history
        history_str = format_history(history)
        
        # Find supporter's next response (target output)
        target_response = ""
        used_strategy = "Unknown"
        
        # Look for next supporter utterance
        for next_idx in range(idx + 1, len(dialog)):
            next_utterance = dialog[next_idx]
            if next_utterance.get('speaker') == 'supporter':
                target_response = next_utterance.get('text', '').strip()
                used_strategy = next_utterance.get('strategy', 'Unknown')
                break
        
        # Skip if no supporter response found
        if not target_response:
            # Add to history and continue
            history.append({
                'speaker': 'seeker',
                'text': user_message
            })
            history = history[-HISTORY_WINDOW:]
            continue
        
        # Create instruction
        instruction = (
            f"You are Aura, an empathetic AI assistant specialized in emotional support. "
            f"Your user is feeling {emotion}. They are saying: '{user_message}'. "
            f"The main reason they feel this way is: '{extracted_cause}'. "
            f"Respond with one of the following strategies: {strategies_str}."
        )
        
        # Create sample
        sample = {
            "instruction": instruction,
            "input": {
                "user_message": user_message,
                "emotion": emotion,
                "cause": extracted_cause,
                "entities": entities_str,
                "history": history_str,
                "problem_type": problem_type
            },
            "output": target_response,
            "metadata": {
                "conversation_id": conversation.get('conv_id', 'unknown'),
                "strategy_used": used_strategy,
                "turn_index": idx
            }
        }
        
        samples.append(sample)
        
        # Add seeker's message to history
        history.append({
            'speaker': 'seeker',
            'text': user_message
        })
        history = history[-HISTORY_WINDOW:]
    
    return samples

print("✅ Core pipeline function created successfully!")

### 2.5 Process Full ESConv Dataset

Run the pipeline over all conversations to generate the hyper-contextual dataset.

In [ ]:
print(f"{'='*80}")
print(f"PROCESSING FULL ESConv DATASET")
print(f"{'='*80}\n")

print(f"Starting pipeline for {len(all_conversations):,} conversations...\n")

# Process all conversations
all_samples = []

for conversation in tqdm(all_conversations, desc="Processing conversations"):
    samples = process_conversation(conversation)
    all_samples.extend(samples)

print(f"\n{'='*80}")
print(f"PROCESSING COMPLETE!")
print(f"{'='*80}")
print(f"Total samples generated: {len(all_samples):,}")
print(f"{'='*80}\n")

### 2.6 Dataset Statistics & Validation

Analyze the generated dataset and display statistics.

In [ ]:
print(f"{'='*80}")
print(f"DATASET STATISTICS")
print(f"{'='*80}\n")

# Count samples by emotion
from collections import Counter

emotions = [s['input']['emotion'] for s in all_samples]
emotion_counts = Counter(emotions)

print("📊 Emotion Distribution:")
for emotion, count in emotion_counts.most_common():
    percentage = (count / len(all_samples)) * 100
    print(f"   {emotion:15s}: {count:6,} ({percentage:5.2f}%)")

print(f"\n{'─'*80}\n")

# Count samples by problem type
problem_types = [s['input']['problem_type'] for s in all_samples]
problem_counts = Counter(problem_types)

print("📊 Problem Type Distribution:")
for prob_type, count in problem_counts.most_common():
    percentage = (count / len(all_samples)) * 100
    print(f"   {prob_type:20s}: {count:6,} ({percentage:5.2f}%)")

print(f"\n{'─'*80}\n")

# Count samples by strategy
strategies = [s['metadata']['strategy_used'] for s in all_samples]
strategy_counts = Counter(strategies)

print("📊 Strategy Distribution:")
for strategy, count in strategy_counts.most_common():
    percentage = (count / len(all_samples)) * 100
    print(f"   {strategy:30s}: {count:6,} ({percentage:5.2f}%)")

print(f"\n{'='*80}")

### 2.7 Display Sample Examples

Show 3 complete examples from the generated dataset.

In [ ]:
print(f"\n{'='*80}")
print(f"SAMPLE EXAMPLES FROM GENERATED DATASET")
print(f"{'='*80}\n")

# Display 3 random samples
import random
sample_indices = random.sample(range(len(all_samples)), min(3, len(all_samples)))

for idx, sample_idx in enumerate(sample_indices, 1):
    sample = all_samples[sample_idx]
    
    print(f"Example {idx}:")
    print(f"{'─'*80}")
    print(f"\n📝 Instruction:")
    print(f"   {sample['instruction']}\n")
    
    print(f"📥 Input:")
    for key, value in sample['input'].items():
        print(f"   {key:15s}: {value}")
    
    print(f"\n📤 Output:")
    print(f"   {sample['output']}\n")
    
    print(f"🔖 Metadata:")
    for key, value in sample['metadata'].items():
        print(f"   {key:15s}: {value}")
    
    print(f"\n{'='*80}\n")

print("✅ Sample examples displayed successfully!")

### 2.8 Save Dataset and Create Splits

Save the complete dataset and split into train (90%) and validation (10%).

In [ ]:
import random

# Set random seed
random.seed(42)

# Shuffle dataset
shuffled_samples = all_samples.copy()
random.shuffle(shuffled_samples)

# Calculate split
total_samples = len(shuffled_samples)
train_size = int(0.9 * total_samples)

# Split data
llm_train = shuffled_samples[:train_size]
llm_val = shuffled_samples[train_size:]

print(f"{'='*80}")
print(f"DATASET SPLITTING")
print(f"{'='*80}")
print(f"Total samples:      {total_samples:,}")
print(f"Training samples:   {len(llm_train):,} ({len(llm_train)/total_samples*100:.1f}%)")
print(f"Validation samples: {len(llm_val):,} ({len(llm_val)/total_samples*100:.1f}%)")
print(f"{'='*80}\n")

# Create output directory
output_dir = Path("llm_training_data")
output_dir.mkdir(exist_ok=True)

# Save complete dataset
full_data_path = output_dir / "llm_training_data.json"
with open(full_data_path, 'w', encoding='utf-8') as f:
    json.dump(shuffled_samples, f, indent=2, ensure_ascii=False)

print(f"✅ Saved complete dataset to '{full_data_path}' ({total_samples:,} samples)\n")

# Save train/val splits
train_path = output_dir / "llm_train.json"
val_path = output_dir / "llm_val.json"

with open(train_path, 'w', encoding='utf-8') as f:
    json.dump(llm_train, f, indent=2, ensure_ascii=False)

with open(val_path, 'w', encoding='utf-8') as f:
    json.dump(llm_val, f, indent=2, ensure_ascii=False)

print(f"✅ Saved training set to '{train_path}' ({len(llm_train):,} samples)")
print(f"✅ Saved validation set to '{val_path}' ({len(llm_val):,} samples)")

print(f"\n{'='*80}")
print(f"ALL FILES SAVED SUCCESSFULLY!")
print(f"{'='*80}")

### 2.9 Final Verification

Verify saved files and provide completion summary.

In [ ]:
# Verify saved files
print(f"{'='*80}")
print(f"FILE VERIFICATION")
print(f"{'='*80}\n")

# Load and verify each file
with open(full_data_path, 'r', encoding='utf-8') as f:
    loaded_full = json.load(f)
    print(f"✅ llm_training_data.json: {len(loaded_full):,} samples")

with open(train_path, 'r', encoding='utf-8') as f:
    loaded_train = json.load(f)
    print(f"✅ llm_train.json: {len(loaded_train):,} samples")

with open(val_path, 'r', encoding='utf-8') as f:
    loaded_val = json.load(f)
    print(f"✅ llm_val.json: {len(loaded_val):,} samples")

print(f"\n{'='*80}")
print(f"📊 FINAL PIPELINE SUMMARY")
print(f"{'='*80}\n")

print("✅ Part 1: ECE Model Inference Testing (Day 9)")
print(f"   - Loaded trained ECE model from Week 9")
print(f"   - Created find_cause() inference function")
print(f"   - Tested on 10 example sentences")
print(f"   - Implemented fallback mechanism\n")

print("✅ Part 2: Hyper-Contextual Dataset Generation (Day 10)")
print(f"   - Loaded {len(all_conversations):,} ESConv conversations")
print(f"   - Integrated ECE model for cause extraction")
print(f"   - Integrated spaCy NER for entity extraction")
print(f"   - Maintained sliding window context (3 turns)")
print(f"   - Inferred problem types automatically")
print(f"   - Generated {len(all_samples):,} instruction-tuning samples\n")

print("✅ Dataset Enrichment:")
print(f"   - Emotion labels from ESConv")
print(f"   - Extracted causes via ECE model")
print(f"   - Named entities via spaCy NER")
print(f"   - Conversation history (sliding window)")
print(f"   - Problem type classification")
print(f"   - Support strategy labels\n")

print("✅ Output Files:")
print(f"   - llm_training_data/llm_training_data.json (complete dataset)")
print(f"   - llm_training_data/llm_train.json (90% training)")
print(f"   - llm_training_data/llm_val.json (10% validation)")

print(f"\n{'='*80}")
print(f"🎉 HYPER-CONTEXTUAL DATASET GENERATION COMPLETE!")
print(f"{'='*80}")
print(f"\n✨ Ready for Week 10 LLM Fine-Tuning")
print(f"   Dataset: {len(all_samples):,} instruction-tuning samples")
print(f"   Next: Fine-tune LLM (GPT-2, LLaMA, or similar) on this dataset\n")

---
## 🎯 Next Steps: Week 10 LLM Fine-Tuning

### LLM Fine-Tuning Pipeline

**1. Choose Base Model**
- GPT-2 (small, medium, large)
- LLaMA 2 (7B, 13B)
- Mistral 7B
- Phi-2

**2. Prepare for Instruction Tuning**
- Load `llm_train.json` and `llm_val.json`
- Format as prompt-completion pairs
- Implement custom DataLoader

**3. Fine-Tuning Strategy**
- LoRA (Low-Rank Adaptation) for efficiency
- QLoRA (Quantized LoRA) for larger models
- Full fine-tuning for smaller models

**4. Training Configuration**
- Learning rate: 2e-5
- Batch size: 4-8
- Epochs: 3-5
- Warmup steps: 100
- Gradient accumulation: 4

**5. Evaluation Metrics**
- Perplexity
- BLEU score
- ROUGE score
- Human evaluation (empathy, coherence)

**6. Integration**
- Deploy fine-tuned model
- Create API endpoint
- Integrate with Aura chat orchestrator

---

## 📝 Documentation

**Files Created:**
- `llm_training_data/llm_training_data.json` - Complete hyper-contextual dataset
- `llm_training_data/llm_train.json` - Training split (90%)
- `llm_training_data/llm_val.json` - Validation split (10%)

**Dataset Schema:**
```json
{
  "instruction": "You are Aura, an empathetic AI assistant...",
  "input": {
    "user_message": "...",
    "emotion": "sad",
    "cause": "...",
    "entities": "...",
    "history": "...",
    "problem_type": "relationship"
  },
  "output": "[Empathetic supporter response]",
  "metadata": {
    "conversation_id": "...",
    "strategy_used": "Affirmation and Reassurance",
    "turn_index": 3
  }
}
```

**Key Features:**
- ECE-powered cause extraction
- NER-based entity recognition
- Context-aware with sliding window history
- Problem type classification
- Strategy-labeled responses

---

**Pipeline Completed: November 16, 2025**  
**Status: ✅ Ready for LLM Fine-Tuning**

---